# Adaptive Information Routing for Medical Image Segmentation
### Thesis Experiments - Kaggle Notebook

**Trước khi chạy:**
1. Vào **Settings** (panel bên phải) => **Accelerator** => chọn **GPU T4 x2** hoặc **P100**
2. **Add Data** => tìm Kaggle Dataset chứa thư mục `data/` đã upload => Add
3. Điền `GITHUB_REPO` và `KAGGLE_DATASET_SLUG` ở cell cấu hình bên dưới
4. **Run All** hoặc chạy tuần tự từng section

**Output** lưu tại `/kaggle/working/` và có thể tải về sau khi notebook hoàn thành.

---
## 0. Cấu hình

In [ ]:
# ── Repo ───────────────────────────────────────────────────────────────
GITHUB_REPO  = "YOUR_USERNAME/YOUR_REPO"   # vd: username/hutech-thesis-01
GITHUB_TOKEN = ""   # để trống nếu repo public; điền PAT nếu repo private

# ── Kaggle Dataset chứa data/ ─────────────────────────────────────────
# Slug lấy từ URL: kaggle.com/datasets/<owner>/<slug>
KAGGLE_DATASET_SLUG = "YOUR_USERNAME/thesis-medical-segmentation-data"

# Cấu trúc bên trong dataset trên Kaggle phải là:
#   /kaggle/input/<slug>/isic/images/*.jpg
#   /kaggle/input/<slug>/isic/masks/*.png
#   /kaggle/input/<slug>/glas/train/images/*.bmp
#   /kaggle/input/<slug>/covid/images/*.png   (COVID-QU-Ex, 2913 ảnh X-ray)
#   /kaggle/input/<slug>/covid/masks/*.png    (infection masks, tên file khớp)
#   /kaggle/input/<slug>/lung/train/images/*.png
#   /kaggle/input/<slug>/dsb2018/images/*.png
#
# COVID dataset: dùng prepare_covidqu.py để extract từ covidqu.zip
#   python prepare_covidqu.py --zip covidqu.zip --out data/covid
#   Kaggle source: anasmohammedtahir/covidqu (Infection Segmentation Data)

# ── Dataset ────────────────────────────────────────────────────────────
# Chọn 1 dataset để chạy ablation study
# 'glas'    => nhỏ nhất,  ~1.5 giờ/4 models trên T4
# 'isic'    => trung bình, ~5 giờ/4 models
# 'dsb2018' => trung bình, ~4 giờ/4 models
# 'lung'    => nhỏ,       ~1 giờ/4 models
# 'covid'   => 2913 ảnh,  ~6 giờ/4 models (256×256, X-ray COVID-QU-Ex)
DATASET = "glas"

# ── Training ───────────────────────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 4
NUM_WORKERS = 2

# ── Paths ──────────────────────────────────────────────────────────────
KAGGLE_INPUT = f"/kaggle/input/{KAGGLE_DATASET_SLUG.split('/')[-1]}"
DATA_ROOT    = f"{KAGGLE_INPUT}/{DATASET}"
REPO_DIR     = "/kaggle/working/repo"
CKPT_DIR     = "/kaggle/working/checkpoints"
RESULT_DIR   = "/kaggle/working/results"

print(f"Dataset    : {DATASET}")
print(f"Data root  : {DATA_ROOT}")
print(f"Epochs     : {EPOCHS}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Repo dir   : {REPO_DIR}")

---
## 1. Setup: GPU · Clone Repo · Dependencies

In [ ]:
# ── Kiểm tra GPU ──────────────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "GPU chưa được bật. Vào Settings => Accelerator => GPU."

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}  ({gpu_mem:.0f} GB VRAM)")
print(f"CUDA : {torch.version.cuda}  |  PyTorch : {torch.__version__}")

In [ ]:
# ── Clone repo từ GitHub ───────────────────────────────────────────────
import os, shutil

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_url = (
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    if GITHUB_TOKEN
    else f"https://github.com/{GITHUB_REPO}.git"
)

!git clone --depth 1 {clone_url} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
# torch, torchvision, numpy, matplotlib, scipy đã có sẵn trên Kaggle
!pip install -q tqdm

import os
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
print("Setup hoàn tất.")
print(f"Working dir : {os.getcwd()}")

---
## 2. Kiểm tra Dataset từ Kaggle Input

In [ ]:
# ── Tự động tìm data root bất kể cấu trúc thư mục Kaggle ─────────────
# Kaggle có thể mount dataset theo nhiều cấu trúc khác nhau:
#   /kaggle/input/<slug>/<dataset>/
#   /kaggle/input/datasets/<owner>/<slug>/data/<dataset>/   ← trường hợp thực tế
#   /kaggle/input/<slug>/data/<dataset>/
# → dùng rglob để tìm tự động, không phụ thuộc vào slug hay cấu trúc cụ thể
from pathlib import Path

kaggle_input = Path("/kaggle/input")

print("Datasets đã mount:")
for d in kaggle_input.iterdir():
    print(f"  {d}")

# Tìm thư mục tên đúng DATASET ở bất kỳ độ sâu nào
hits = [p for p in kaggle_input.rglob(DATASET) if p.is_dir()]

if not hits:
    # In cây thư mục để debug
    print("\nCây thư mục /kaggle/input (30 dòng đầu):")
    for p in sorted(kaggle_input.rglob("*"))[:30]:
        print(f"  {p}")
    raise FileNotFoundError(
        f"Không tìm thấy thư mục '{DATASET}' trong /kaggle/input/.\n"
        f"Kiểm tra lại cấu trúc thư mục trong Kaggle dataset."
    )

# Ưu tiên thư mục nào có images/ hoặc train/ bên trong
def _is_valid(p):
    return (p / "images").exists() or (p / "train").exists()

valid = [p for p in hits if _is_valid(p)]
data_root = valid[0] if valid else hits[0]
DATA_ROOT = str(data_root)

print(f"\nDataset '{DATASET}' tìm thấy tại:\n  {data_root}")

In [ ]:
# ── Xác nhận cấu trúc thư mục theo từng dataset ───────────────────────
from pathlib import Path

STRUCTURE_CHECKS = {
    "isic"   : {"images": "*.jpg", "masks": "*.png"},
    "glas"   : {
        "train/images": "*.bmp", "train/masks": "*.bmp",
        "test/images" : "*.bmp", "test/masks" : "*.bmp",
    },
    "covid"  : {"images": "*.png", "masks": "*.png"},
    "lung"   : {
        "train/images": "*.png", "train/masks": "*.png",
        "test/images" : "*.png", "test/masks" : "*.png",
    },
    "dsb2018": {"images": "*.png", "masks": "*.png"},
}

root   = Path(DATA_ROOT)
checks = STRUCTURE_CHECKS.get(DATASET, {"images": "*.*", "masks": "*.*"})

print(f"Dataset: {DATASET}  =>  {root}")
all_ok = True
for sub, pat in checks.items():
    d = root / sub
    n = len(list(d.glob(pat))) if d.exists() else 0
    ok = "✓" if n > 0 else "✗"
    print(f"  {ok} {sub:<22} {n:>5} files")
    if n == 0:
        all_ok = False

if not all_ok:
    raise RuntimeError(
        "Dataset chưa đầy đủ. Kiểm tra lại cấu trúc thư mục trong Kaggle dataset."
    )
print("\nDataset sẵn sàng.")

---
## 3. Kiểm tra Kiến trúc Mô hình

In [ ]:
import torch
from models import TransAttUnet, ProposedModel, UNet, AttUNet
from datasets import DATASET_DEFAULTS

cfg = DATASET_DEFAULTS[DATASET]
H   = cfg["img_size"]
C   = cfg["n_channels"]
x   = torch.randn(2, C, H, H)

configs = {
    "unet":      UNet(C, 1),
    "att_unet":  AttUNet(C, 1),
    "baseline":  TransAttUnet(C, 1),
    "adar_only": ProposedModel(C, 1, use_adar=True,  use_csr=False),
    "csr_only":  ProposedModel(C, 1, use_adar=False, use_csr=True),
    "proposed":  ProposedModel(C, 1, use_adar=True,  use_csr=True),
}

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Input: (2, {C}, {H}, {H})\n")
print(f"{'Model':<15} {'Params':>10}  Output")
print("-" * 40)
for name, model in configs.items():
    out    = model(x)
    logits = out[0] if isinstance(out, tuple) else out
    print(f"{name:<15} {count_params(model)/1e6:>9.2f}M  {tuple(logits.shape)}")

print("\nTất cả mô hình OK.")

---
## 4. Training - 6 Mô hình

| # | Config | ADAR | CSR | Mục đích |
|---|--------|------|-----|----------|
| 1 | `unet`      | ✗ | ✗ | Baseline U-Net [4] |
| 2 | `att_unet`  | ✗ | ✗ | Baseline Att-UNet [10] |
| 3 | `baseline`  | ✗ | ✗ | Reproduce TransAttUnet_R |
| 4 | `adar_only` | ✓ | ✗ | Đánh giá ADAR đơn lẻ |
| 5 | `csr_only`  | ✗ | ✓ | Đánh giá CSR đơn lẻ |
| 6 | `proposed`  | ✓ | ✓ | Mô hình đầy đủ |

In [ ]:
# ── Helper: chạy training ──────────────────────────────────────────────
import subprocess, sys

# Thứ tự: 2 baseline trước, rồi 4 ablation variants
ALL_MODELS     = ["unet", "att_unet", "baseline", "adar_only", "csr_only", "proposed"]
ABLATION_MODELS = ALL_MODELS  # dùng chung cho eval + bảng

def run_train(model_name):
    print(f"\n{'='*56}")
    print(f"  Training: {model_name.upper()}  |  Dataset: {DATASET.upper()}")
    print(f"{'='*56}")
    cmd = [
        sys.executable, "train.py",
        "--dataset",     DATASET,
        "--data_root",   DATA_ROOT,
        "--model",       model_name,
        "--epochs",      str(EPOCHS),
        "--batch_size",  str(BATCH_SIZE),
        "--lr",          "0.01",          # SGD lr=0.01, paper setting
        "--save_dir",    CKPT_DIR,
        "--num_workers", str(NUM_WORKERS),
        "--seed",        "42",
    ]
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print(f"[ERROR] Training {model_name} thất bại (exit {result.returncode}).")
        return False
    return True

In [ ]:
%%time
run_train("unet")

In [ ]:
%%time
run_train("att_unet")

In [ ]:
%%time
run_train("baseline")

In [ ]:
%%time
run_train("adar_only")

In [ ]:
%%time
run_train("csr_only")

In [ ]:
%%time
run_train("proposed")

In [ ]:
# ── Xác nhận checkpoint ────────────────────────────────────────────────
from pathlib import Path

missing = []
for m in ABLATION_MODELS:
    ckpt   = Path(CKPT_DIR) / DATASET / m / "best_model.pth"
    size   = f"{ckpt.stat().st_size/1e6:.1f} MB" if ckpt.exists() else "MISSING"
    status = "✓" if ckpt.exists() else "✗"
    print(f"  {status}  {m:<12}  {size}")
    if not ckpt.exists():
        missing.append(m)

if missing:
    print(f"\n[!] Thiếu checkpoint: {missing}")
else:
    print("\nTất cả checkpoint sẵn sàng.")

---
## 5. Evaluation

In [ ]:
import subprocess, sys
from pathlib import Path

def run_evaluate(model_name):
    ckpt = str(Path(CKPT_DIR) / DATASET / model_name / "best_model.pth")
    if not Path(ckpt).exists():
        print(f"[skip] {model_name}: checkpoint không có")
        return
    print(f"\n[Evaluating] {model_name.upper()}")
    cmd = [
        sys.executable, "evaluate.py",
        "--dataset",    DATASET,
        "--data_root",  DATA_ROOT,
        "--model",      model_name,
        "--checkpoint", ckpt,
        "--output_dir", RESULT_DIR,
        "--n_viz",      "6",
    ]
    subprocess.run(cmd, text=True)

for m in ABLATION_MODELS:
    run_evaluate(m)

---
## 6. Ablation Table

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

METRIC_KEYS = ["dice", "iou", "acc", "rec", "pre", "hd95"]
LABELS = {
    "unet":      "U-Net [4]",
    "att_unet":  "Att-UNet [10]",
    "baseline":  "TransAttUnet_R (baseline)",
    "adar_only": "+ ADAR",
    "csr_only":  "+ CSR",
    "proposed":  "+ ADAR + CSR (Proposed)",
}

rows = []
for m in ABLATION_MODELS:
    summary_path = Path(RESULT_DIR) / DATASET / m / "summary.txt"
    if not summary_path.exists():
        continue
    row = {"Model": LABELS.get(m, m)}
    for line in summary_path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[0].lower() in METRIC_KEYS:
            row[parts[0].upper()] = f"{float(parts[1]):.2f}"
    rows.append(row)

if rows:
    df = pd.DataFrame(rows).set_index("Model")

    def highlight_best(col):
        ascending = col.name == "HD95"
        vals = pd.to_numeric(col, errors="coerce")
        best = vals.min() if ascending else vals.max()
        return ["font-weight: bold; color: #1a7a1a" if v == best else "" for v in vals]

    display(df.style.apply(highlight_best))
    out_csv = Path(RESULT_DIR) / DATASET / "ablation_table.csv"
    df.to_csv(out_csv)
    print(f"\nBảng lưu tại: {out_csv}")
else:
    print("Chưa có kết quả. Chạy Section 5 trước.")

---
## 7. Visualizations

### 7.1 Segmentation Grid

In [ ]:
import torch, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from datasets import DATASET_REGISTRY, DATASET_DEFAULTS
from train import build_model

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg    = DATASET_DEFAULTS[DATASET]

test_ds = DATASET_REGISTRY[DATASET](
    DATA_ROOT, split="test", img_size=cfg["img_size"], augment=False
)
# num_workers=0 trên CPU để tránh deadlock (DataLoader spawn warning on macOS)
_nw = NUM_WORKERS if torch.cuda.is_available() else 0
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=_nw)

def load_model(name):
    m    = build_model(name, cfg["n_channels"], 1)
    ckpt = Path(CKPT_DIR) / DATASET / name / "best_model.pth"
    state = torch.load(ckpt, map_location="cpu")
    m.load_state_dict(state.get("model_state", state))
    return m.to(device).eval()

models_loaded = {}
for name in ABLATION_MODELS:
    if (Path(CKPT_DIR) / DATASET / name / "best_model.pth").exists():
        models_loaded[name] = load_model(name)
        print(f"  ✓ Loaded {name}")

print(f"\n{len(models_loaded)}/{len(ABLATION_MODELS)} models loaded.")

In [ ]:
N_SAMPLES   = 4
model_names = list(models_loaded.keys())
n_cols      = 2 + len(model_names)
col_titles  = ["Input", "Ground Truth"] + [LABELS.get(m, m) for m in model_names]

fig, axes = plt.subplots(N_SAMPLES, n_cols, figsize=(3 * n_cols, 3.2 * N_SAMPLES))
axes = np.array(axes)

for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=8, fontweight="bold", pad=4)

@torch.no_grad()
def predict(model, img_tensor):
    out    = model(img_tensor)
    logits = out[0] if isinstance(out, tuple) else out
    return torch.sigmoid(logits)[0, 0].cpu().numpy()

for i, (imgs, masks, _) in enumerate(test_loader):
    if i >= N_SAMPLES:
        break
    imgs_dev = imgs.to(device)
    img_np   = np.clip(imgs[0].permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1)

    axes[i, 0].imshow(img_np if img_np.shape[-1] == 3 else img_np[..., 0], cmap="gray" if img_np.shape[-1] != 3 else None)
    axes[i, 0].axis("off")
    axes[i, 1].imshow(masks[0, 0].numpy(), cmap="gray")
    axes[i, 1].axis("off")

    for j, name in enumerate(model_names):
        pred = predict(models_loaded[name], imgs_dev)
        axes[i, 2 + j].imshow((pred > 0.5).astype(float), cmap="gray")
        axes[i, 2 + j].axis("off")

plt.suptitle(f"Segmentation Results - {DATASET.upper()}", fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
save_path = Path(RESULT_DIR) / DATASET / "segmentation_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved => {save_path}")

### 7.2 ADAR Routing Weights

In [ ]:
adar_name = next((m for m in ["proposed", "adar_only"] if m in models_loaded), None)

if adar_name is None:
    print("Cần proposed hoặc adar_only checkpoint.")
else:
    adar_model = models_loaded[adar_name]
    weights = []
    with torch.no_grad():
        for imgs, _, _ in test_loader:
            out = adar_model(imgs.to(device))
            if isinstance(out, tuple) and out[1] and out[1].get("adar") is not None:
                weights.append(out[1]["adar"].cpu().numpy())

    if not weights:
        print("Không có ADAR weights trong output. Kiểm tra model.")
    else:
        import numpy as np
        all_w  = np.concatenate(weights, axis=0)
        labels = ["w_tsa\n(channel)", "w_gsa\n(spatial)", "w_orig\n(residual)"]
        colors = ["#4C72B0", "#DD8452", "#55A868"]
        means  = all_w.mean(axis=0)
        stds   = all_w.std(axis=0)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

        bp = ax1.boxplot([all_w[:, k] for k in range(3)], labels=labels,
                         patch_artist=True, widths=0.5,
                         medianprops=dict(color="red", linewidth=2))
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c); patch.set_alpha(0.65)
        ax1.axhline(1/3, color="gray", ls="--", lw=1, label="Uniform (1/3)")
        ax1.set_ylim(0, 1); ax1.set_ylabel("Routing weight")
        ax1.set_title(f"ADAR weight distribution ({adar_name}, n={len(all_w)})", fontweight="bold")
        ax1.legend(fontsize=9)

        bars = ax2.bar(range(3), means, yerr=stds, capsize=5, color=colors, alpha=0.8, width=0.5)
        ax2.set_xticks(range(3)); ax2.set_xticklabels(labels)
        ax2.axhline(1/3, color="gray", ls="--", lw=1)
        ax2.set_ylim(0, 1); ax2.set_ylabel("Mean weight")
        ax2.set_title("Mean ± Std routing weights", fontweight="bold")
        for bar, m_ in zip(bars, means):
            ax2.text(bar.get_x() + bar.get_width()/2, m_ + 0.03,
                     f"{m_:.3f}", ha="center", fontsize=9, fontweight="bold")

        plt.tight_layout()
        save_path = Path(RESULT_DIR) / DATASET / "adar_routing_weights.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved => {save_path}")
        print(f"w_tsa={means[0]:.3f}  w_gsa={means[1]:.3f}  w_orig={means[2]:.3f}")

### 7.3 CSR Scale Weights

In [ ]:
csr_name = next((m for m in ["proposed", "csr_only"] if m in models_loaded), None)

if csr_name is None:
    print("Cần proposed hoặc csr_only checkpoint.")
else:
    csr_model = models_loaded[csr_name]
    csr_acc   = {f"csr{i}": [] for i in range(1, 5)}

    with torch.no_grad():
        for imgs, _, _ in test_loader:
            out = csr_model(imgs.to(device))
            if isinstance(out, tuple) and out[1]:
                for k in csr_acc:
                    if out[1].get(k) is not None:
                        csr_acc[k].append(out[1][k].cpu().numpy())

    if not all(len(v) > 0 for v in csr_acc.values()):
        print("Không có CSR weights trong output.")
    else:
        import numpy as np
        stage_labels = ["Stage 1\n(deep)", "Stage 2", "Stage 3", "Stage 4\n(shallow)"]
        colors = ["#4C72B0", "#DD8452"]
        means_p, means_c, stds_p, stds_c = [], [], [], []
        for k in ["csr1", "csr2", "csr3", "csr4"]:
            arr = np.concatenate(csr_acc[k], axis=0)
            means_p.append(arr[:, 0].mean()); stds_p.append(arr[:, 0].std())
            means_c.append(arr[:, 1].mean()); stds_c.append(arr[:, 1].std())

        x, w = np.arange(4), 0.35
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

        ax1.bar(x - w/2, means_p, w, yerr=stds_p, capsize=4,
                label="w_prev (semantic)", color=colors[0], alpha=0.8)
        ax1.bar(x + w/2, means_c, w, yerr=stds_c, capsize=4,
                label="w_cur (boundary)", color=colors[1], alpha=0.8)
        ax1.set_xticks(x); ax1.set_xticklabels(stage_labels)
        ax1.axhline(0.5, color="gray", ls="--", lw=1)
        ax1.set_ylim(0, 1); ax1.set_ylabel("Mean routing weight")
        ax1.set_title(f"CSR scale routing per decoder stage ({csr_name})", fontweight="bold")
        ax1.legend(fontsize=9)

        ax2.bar(x, means_p, color=colors[0], alpha=0.8, label="w_prev")
        ax2.bar(x, means_c, bottom=means_p, color=colors[1], alpha=0.8, label="w_cur")
        for xi, (mp, mc) in enumerate(zip(means_p, means_c)):
            ax2.text(xi, mp/2,      f"{mp:.2f}", ha="center", color="white", fontsize=9, fontweight="bold")
            ax2.text(xi, mp + mc/2, f"{mc:.2f}", ha="center", color="white", fontsize=9, fontweight="bold")
        ax2.set_xticks(x); ax2.set_xticklabels(stage_labels)
        ax2.set_ylim(0, 1.05); ax2.set_ylabel("Proportion")
        ax2.set_title("Relative scale preference", fontweight="bold")
        ax2.legend(fontsize=9)

        plt.tight_layout()
        save_path = Path(RESULT_DIR) / DATASET / "csr_stage_weights.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved => {save_path}")


### 7.4 Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from pathlib import Path

n_models = len(ABLATION_MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
axes = np.array(axes).flatten()

for i, name in enumerate(ABLATION_MODELS):
    curve_path = Path(CKPT_DIR) / DATASET / name / "training_curves.png"
    if curve_path.exists():
        axes[i].imshow(mpimg.imread(str(curve_path)))
        axes[i].set_title(LABELS.get(name, name), fontweight="bold", fontsize=10)
    else:
        axes[i].text(0.5, 0.5, f"{name}\nnot found",
                     ha="center", va="center", transform=axes[i].transAxes, color="gray")
    axes[i].axis("off")

# Hide unused subplots
for j in range(n_models, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"Training Curves - {DATASET.upper()}", fontsize=12, fontweight="bold")
plt.tight_layout()
save_path = Path(RESULT_DIR) / DATASET / "all_training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved => {save_path}")


---
## 8. Tổng hợp & Xuất kết quả

In [ ]:
# ── In bảng tổng hợp cuối cùng ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

csv_path = Path(RESULT_DIR) / DATASET / "ablation_table.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path, index_col="Model")
    print(f"\n{'='*60}")
    print(f"  ABLATION RESULTS - {DATASET.upper()}  ({EPOCHS} epochs)")
    print(f"{'='*60}")
    print(df.to_string())

# ── Liệt kê tất cả output files ────────────────────────────────────────
result_base = Path(RESULT_DIR) / DATASET
print(f"\nFiles trong {result_base}:")
for f in sorted(result_base.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(result_base)):<45} {f.stat().st_size/1024:6.0f} KB")

In [ ]:
# ── Zip toàn bộ kết quả => /kaggle/working/ để tải về ──────────────────
import shutil
from pathlib import Path

zip_name = f"thesis_results_{DATASET}"
zip_out  = Path("/kaggle/working") / zip_name

# Zip checkpoints + results
bundle_dir = Path("/kaggle/working/_bundle")
bundle_dir.mkdir(exist_ok=True)
shutil.copytree(Path(CKPT_DIR)   / DATASET, bundle_dir / "checkpoints" / DATASET, dirs_exist_ok=True)
shutil.copytree(Path(RESULT_DIR) / DATASET, bundle_dir / "results"     / DATASET, dirs_exist_ok=True)

shutil.make_archive(str(zip_out), "zip", root_dir=bundle_dir)
shutil.rmtree(bundle_dir)

zip_file = Path(str(zip_out) + ".zip")
size_mb  = zip_file.stat().st_size / 1e6
print(f"\nZip tạo tại: {zip_file}  ({size_mb:.1f} MB)")
print("Tải về: Kaggle => Output => Download")